# Workshop 3 · 04 — Data Science auf unstrukturierten Daten (Aufgabe)

**Ziel:** Aus Text **Embeddings** erzeugen und darauf Data-Science-Verfahren anwenden: Clustering, Anomalie-Erkennung, Similarity-Suche.

**Vorgegeben sind Plumbing und Hyperparameter.** Offen bleiben die eigentlichen Aufrufe: `ai.embed`, `KMeans`, `IsolationForest`, die Cosine-Suche und die Cluster-Benennung. Setze sie an den `# TODO`-Stellen ein.

> Zentrales Prinzip: **Text → Vektoren (`ai.embed`) → klassisches ML.**

## Schritt 0 — Dokument bauen und embedden
Das "Dokument" je Meldung ist vorgegeben. **Aufgabe:** Wende `ai.embed` an -> Spalte `vector`.

Signatur: `docs.ai.embed(input_col="doc", output_col="vector")`

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.spark.aifunc as aifunc
from pyspark.sql.functions import concat_ws, col

base = spark.table("asset_clean")

# Vorgegeben: "Dokument" je Meldung
docs = base.withColumn("doc", concat_ws(". ", col("komponente"), col("freitext_meldung"), col("kunden_kommentar")))

# TODO: ai.embed anwenden -> Spalte "vector"
emb = ...   # <-- hier den ai.embed-Aufruf einsetzen

# Vorgegeben: nach pandas holen
pdf = emb.select("meldung_id", "depot_norm", "komponente", "schweregrad",
                 "freitext_meldung", "vector").toPandas()
print("Meldungen x Vektordim:", pdf.shape, "/", len(pdf["vector"].iloc[0]))

## 1) Clustering — Themen automatisch finden
Die Vektor-Matrix `X` ist vorgegeben. **Aufgabe:** Wende `KMeans` an und schreibe das Label nach `pdf["cluster"]`.

Signatur: `KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)`

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# Vorgegeben: Vektoren zu Matrix X stapeln
X = np.vstack(pdf["vector"].apply(lambda v: np.array(v, dtype=float)).values)

k = 4
# TODO: KMeans auf X anwenden und Cluster-Label speichern
pdf["cluster"] = ...   # <-- hier KMeans(...).fit_predict(X) einsetzen

display(pdf.groupby("cluster").size().reset_index(name="anzahl"))

### (Bonus) Cluster automatisch benennen
Das Gerüst ist vorgegeben. **Aufgabe:** Wende die pandas-AI-Function `ai.generate_response` an -> Spalte `label`.

Signatur: `one.ai.generate_response(prompt="... {beispiele} ...", is_prompt_template=True)`

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.aifunc as aifunc_pd
import pandas as pd

labels = {}
for c in sorted(pdf["cluster"].unique()):
    beispiele = " || ".join(pdf[pdf.cluster == c]["freitext_meldung"].head(4).tolist())
    one = pd.DataFrame({"beispiele": [beispiele]})
    # TODO: pandas-AI-Function anwenden -> Spalte "label" (Prompt-Template auf {beispiele})
    one["label"] = ...   # <-- hier one.ai.generate_response(...) einsetzen
    labels[c] = one["label"].iloc[0]

pdf["thema"] = pdf["cluster"].map(labels)
display(pdf[["cluster", "thema"]].drop_duplicates().sort_values("cluster"))

## 2) Anomalie-Erkennung — ungewöhnliche Meldungen
**Aufgabe:** Wende `IsolationForest` auf `X` an und markiere Anomalien in `pdf["anomalie"]`.

Signatur: `IsolationForest(contamination=0.15, random_state=0).fit_predict(X) == -1`

In [ ]:
from sklearn.ensemble import IsolationForest

# TODO: IsolationForest auf X anwenden -> pdf["anomalie"] (True/False)
pdf["anomalie"] = ...   # <-- hier IsolationForest(...).fit_predict(X) == -1 einsetzen

print("Anomalien:", int(pdf["anomalie"].sum()), "von", len(pdf))
display(pdf[pdf["anomalie"]][["meldung_id", "komponente", "schweregrad", "freitext_meldung"]])

## 3) Similarity-Suche — „finde ähnliche Schäden"
Die Suchlogik (Cosine + Top-k) ist vorgegeben. **Aufgabe:** Vervollständige `embed_query` mit `ai.embed`.

Signatur: `spark.createDataFrame([(q,)], ["doc"]).ai.embed(input_col="doc", output_col="vector").collect()[0]["vector"]`

In [ ]:
# This code uses AI. Always review output for mistakes.

def embed_query(q):
    # TODO: q mit ai.embed zu einem Vektor machen
    row = ...   # <-- hier ai.embed auf einem 1-Zeilen-DataFrame einsetzen
    return np.array(row, dtype=float)

# Vorgegeben: Cosine-Aehnlichkeit gegen X, Top-k
def find_similar(q, top_k=3):
    qv = embed_query(q)
    sims = X @ qv / (np.linalg.norm(X, axis=1) * np.linalg.norm(qv) + 1e-9)
    idx = np.argsort(sims)[::-1][:top_k]
    res = pdf.iloc[idx][["meldung_id", "komponente", "freitext_meldung"]].copy()
    res["aehnlichkeit"] = np.round(sims[idx], 3)
    return res

display(find_similar("Roststelle am Tuerrahmen, klemmende Tuer"))

## Schritt 4 — Ergebnisse persistieren (vorgegeben)
Speichert die angereicherten Meldungen als Tabelle `meldung_ml`. Einfach ausführen.

In [ ]:
out = spark.createDataFrame(
    pdf[["meldung_id", "depot_norm", "komponente", "schweregrad", "cluster", "thema", "anomalie"]])
out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("meldung_ml")
display(out)

### Ergebnis-Check
Aus Text wurden Vektoren (`ai.embed`), darauf liefen **Clustering**, **Anomalie-Erkennung** und **semantische Suche**. Ergebnis `meldung_ml`.

> **Bei großen Korpora:** Vektoren in **Eventhouse/KQL** legen und mit `series_cosine_similarity()` skalierend suchen statt im Treiber.